In [ ]:
# ============================
# 0) Imports + logging
# ============================
from __future__ import annotations

import logging
from pathlib import Path

import pandas as pd
import preprocessing as pp
import warnings
warnings.filterwarnings("ignore")

LOGGER = logging.getLogger(__name__)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

LOGGER = logging.getLogger("run_preprocessing")


# ============================
# 1) Config
# ============================
DATA_DIR = Path("YOUR_DATA_DIR")  # e.g., Path("./data_validate")
REF_ELIX_XLSX = DATA_DIR / "HCUP_ELIXHAUSER_REFERENCE.xlsx"

PATHS = {
    # Vital signs
    "body_temperature": DATA_DIR / "body_temperature.csv",
    "heart_rate": DATA_DIR / "heart_rate.csv",
    "mean_arterial_pressure": DATA_DIR / "mean_arterial_pressure.csv",
    "so2": DATA_DIR / "so2.csv",
    # Labs
    "crp": DATA_DIR / "crp.csv",
    "bili": DATA_DIR / "bili.csv",
    "leua": DATA_DIR / "leua.csv",
    "krea": DATA_DIR / "krea.csv",
    "hb": DATA_DIR / "hb.csv",
    # Static covariates
    "conditions_stay": DATA_DIR / "conditions_stay.csv",
    "conditions_past": DATA_DIR / "conditions_past.csv",
    "patient_info": DATA_DIR / "patient_info.csv",
    # Antibiotics
    "ab_groups": DATA_DIR / "ab_groups.csv",
}

# Feature space must exist in this runner (or import from a config file)
feature_space = [
    # Static Features
    'cond_hemat',
    'cond_leuk',
    'cond_solid',
    'age',
    'elixhauser_score',
    'sex_male',
    'length_stay',
    # Temporal Features
    'time_lag_1_bt_max',            
    'time_lag_2_bt_max',
    'time_lag_1_bt_mean',            
    'time_lag_2_bt_mean',
    'time_lag_1_krea_max',          
    'time_lag_2_krea_max',          
    'time_lag_1_bili_max',          
    'time_lag_2_bili_max',           
    'time_lag_1_leua_max',          
    'time_lag_2_leua_max',                    
    'time_lag_1_crp_max',           
    'time_lag_2_crp_max',                 
    'time_lag_1_hb_max',           
    'time_lag_2_hb_max',                     
    'time_lag_1_heart_rate_max',            
    'time_lag_2_heart_rate_max',
    'time_lag_1_mean_arterial_pressure_max',            
    'time_lag_2_mean_arterial_pressure_max', 
    'time_lag_1_so2_max',            
    'time_lag_2_so2_max', 
    'fever_variability_lag_1',           
    'fever_variability_lag_2', 
    'last_bt_fever_lag_1',
    'fever_diff_lag_1',
    'fever_diff_lag_2',
    'fever_percent_lag_2',
    'fever_percent_lag_1',
    'fever_lag_1_evening',
    'fever_lag_1_afternoon',
    'fever_lag_1_morning',
    'fever_lag_2_evening',
    'fever_lag_2_afternoon',
    'fever_lag_2_morning',
    'fever_points_lag_1',
    'fever_points_lag_2',
    'fever_change_lag_1',
    'fever_change_lag_all',
    'fever_range_lag_1',
    'fever_range_lag_2',
    'trend',
    'fever_lag_1_night',
    'fever_lag_2_night',
    'skewness_lag_all',                     
    'fourier_max_magnitude'              
    ]

# ============================
# 2) Load data
# ============================
def _read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing input file: {path}")
    return pd.read_csv(path)

LOGGER.info("Loading CSV inputs...")
body_temperature = _read_csv(PATHS["body_temperature"])
heart_rate = _read_csv(PATHS["heart_rate"])
mean_arterial_pressure = _read_csv(PATHS["mean_arterial_pressure"])
so2 = _read_csv(PATHS["so2"])

crp = _read_csv(PATHS["crp"])
bili = _read_csv(PATHS["bili"])
leua = _read_csv(PATHS["leua"])
krea = _read_csv(PATHS["krea"])
hb = _read_csv(PATHS["hb"])

conditions_stay = _read_csv(PATHS["conditions_stay"])
conditions_past = _read_csv(PATHS["conditions_past"])
patient_info = _read_csv(PATHS["patient_info"])
ab_groups = _read_csv(PATHS["ab_groups"])

# Standardize sex column name once
if "gender" in patient_info.columns and "sex" not in patient_info.columns:
    patient_info = patient_info.rename(columns={"gender": "sex"})

if not REF_ELIX_XLSX.exists():
    raise FileNotFoundError(f"Missing Elixhauser reference file: {REF_ELIX_XLSX}")


# ============================
# 3) Pipeline execution
# ============================

# Step 1
pre = pp.preprocess_from_dataframes(
    df_temp=body_temperature,
    df_heart_rates=heart_rate,
    df_so2=so2,
    df_bp=mean_arterial_pressure,
    df_crp=crp,
    df_bili=bili,
    df_leua=leua,
    df_krea=krea,
    df_hb=hb,
    df_ab_groups=ab_groups,
    strict=True,
    copy=True,
)
LOGGER.info("Step 1 done: preprocessing complete")

# Step 2
data = pp.create_time_lags_and_merge_features(
    df_temp=pre.temp,
    df_heart_rates=pre.heart_rates,
    df_so2=pre.so2,
    df_bp=pre.bp,
    df_bili=pre.bili,
    df_crp=pre.crp,
    df_hb=pre.hb,
    df_krea=pre.krea,
    df_leua=pre.leua,
    df_ab_groups=pre.ab_groups,
    strict=True,
    copy=True,
)
LOGGER.info("Step 2 done: time lags + asof merges complete")

# Step 3
fourier_features = (
    data.groupby("encounter_id", group_keys=False)
        .apply(pp.create_fourier_features_for_group, strict=False)
)
data = data.merge(fourier_features, left_on="encounter_id", right_index=True, how="left")
LOGGER.info("Step 3 done: Fourier features merged")

# NOTE (publication hygiene): FFT arrays are object columns. Consider keeping only scalar summaries:
# data = data.drop(columns=["fourier_magnitude","fourier_phase","fourier_real_part","fourier_imaginary_part"])

# Step 4 (your current: ffill only)
data_imp = pp.ffill_impute_within_encounter(
    data,
    group_col="encounter_id",
    time_col="recorded_time",
    columns=None,
    exclude=("value",),
    limit=None,
    copy=True,
    strict=True,
)
LOGGER.info("Step 4 done: imputation complete")

# Step 5
data_lag = pp.add_statistical_lag_features(
    data_imp,
    group_col="encounter_id",
    lag1_col="time_lag_1",
    lag2_col="time_lag_2",
    target_lag_col="time_lag_target",
    max_columns=("value", "krea", "bili", "crp", "leua", "hb", "heart_rate", "so2", "mean_arterial_pressure"),
    value_col="value",
    copy=True,
    strict=True,
    drop_missing_required=True,
)
LOGGER.info("Step 5 done: statistical lag features added")

# Step 6
data_fs = pp.create_features(
    data_lag,
    group_col="encounter_id",
    time_col="recorded_time",
    value_col="value",
    lag1_col="time_lag_1",
    lag2_col="time_lag_2",
    target_lag_col="time_lag_target",
    fever_threshold=38.0,
    min_required_bt_max_col="time_lag_1_bt_max",
    copy=True,
    strict=True,
    fillna_value=0.0,
)
LOGGER.info("Step 6 done: further statistical features added")

# Step 7
data_fs_elix = pp.calculate_elixhauser_score(
    df=data_fs,
    df_conds=conditions_past,
    reference_xlsx_path=str(REF_ELIX_XLSX),
    subject_col="subject_reference",
    dx_col="conditions",
    return_indicators=True,
    strict=True,
)
LOGGER.info("Step 7 done: Elixhauser comorbidities merged")

# Step 8
data_fs_static, enriched_full = pp.add_static_covariates(
    data_fs_elix,
    feature_space=feature_space,
    df_conditions=conditions_stay,
    df_bd=patient_info,
    fever_threshold=38.0,
    min_age_years=18,
    copy=True,
    strict=True,
)
LOGGER.info("Step 8 done: static covariates merged; final ML dataset created")


# ============================
# 4) Optional: Split
# ============================

split = pp.do_train_test_split(
    data_fs_static,
    feature_space=feature_space,
    target_col="fever",
    test_size=0.20,
    random_state=23,
    stratify=True,
    scale=False,
    copy=True,
    strict=True,
)

LOGGER.info("Step 9 done: train/test split complete")

X_train, X_test, y_train, y_test = split.X_train, split.X_test, split.y_train, split.y_test


# ============================
# 5) Optional: Save artifacts
# ============================
OUT_DIR = Path("./artifacts")
OUT_DIR.mkdir(parents=True, exist_ok=True)

data_fs_static.to_parquet(OUT_DIR / "data_fs_static.parquet", index=False)
X_train.to_parquet(OUT_DIR / "X_train.parquet")
X_test.to_parquet(OUT_DIR / "X_test.parquet")
y_train.to_frame("fever").to_parquet(OUT_DIR / "y_train.parquet")
y_test.to_frame("fever").to_parquet(OUT_DIR / "y_test.parquet")

LOGGER.info("Saved artifacts to %s", OUT_DIR.resolve())
